# LoRA-A Training: Q -> A (Question Only)

This notebook trains the LoRA-A adapter on Kaggle's 2xT4 GPUs.
LoRA-A is the realistic baseline that many teams use.

**Prerequisites:**
- Upload this repo to Kaggle as a dataset
- Enable GPU accelerator (2xT4)
- Connect to W&B (optional)

## 1. Setup

In [ ]:
# Install dependencies
!pip install -q unsloth transformers peft bitsandbytes wandb datasets trl accelerate
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
# Copy repo files to working directory (adjust path to your Kaggle dataset)
!cp -r /kaggle/input/lora-rag-clinical/* /kaggle/working/
%cd /kaggle/working

## 2. Configuration

In [ ]:
# Training configuration
SEED = 42  # Change to 123 or 456 for other runs
OUTPUT_DIR = "/kaggle/working/results/lora_a"
WANDB_PROJECT = "lora-rag-clinical"

# Set to True to skip W&B logging
OFFLINE = False

In [ ]:
# Optional: Login to W&B
if not OFFLINE:
    import wandb
    wandb.login()

## 3. Validation Run (100 steps)

Before committing to a full epoch, run 100 steps to verify:
- Loss curve is decreasing (not flat, not exploding)
- No OOM errors
- Gradient norms are reasonable

In [ ]:
# Validation run: 100 steps with frequent logging
from src.training.common import (
    set_seed, load_config, load_training_data,
    setup_model_and_tokenizer, apply_lora,
    format_for_trainer
)
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

# Set seed
set_seed(SEED)

# Load config and data
config = load_config("configs/lora_a.yaml")
examples = load_training_data("data/training/lora_a_train.jsonl")
print(f"Loaded {len(examples)} examples")

# Setup model with Unsloth
model, tokenizer = setup_model_and_tokenizer(
    model_name=config["model"]["name"],
    max_seq_length=config["model"]["max_seq_len"],
    use_unsloth=True,
)

# Apply LoRA
model = apply_lora(model, config, use_unsloth=True)

# Prepare dataset
dataset = format_for_trainer(examples, tokenizer, config["model"]["max_seq_len"])

# 100-step validation run
validation_args = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}/validation",
    max_steps=100,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=config["training"]["learning_rate"],
    logging_steps=5,
    save_steps=10000,  # Don't save during validation
    seed=SEED,
    bf16=True,
    optim="adamw_8bit",
    report_to="wandb" if not OFFLINE else "none",
    run_name=f"lora_a_validation_seed_{SEED}",
    remove_unused_columns=False,
)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True, return_tensors="pt")

trainer = Trainer(
    model=model,
    args=validation_args,
    train_dataset=dataset,
    data_collator=data_collator,
)

print("Starting 100-step validation run...")
trainer.train()
print("Validation complete. Inspect loss curve above before proceeding.")

## 4. Full Training Run

After verifying the validation run looks good, run the full epoch.

In [ ]:
# Full training run
# Note: Re-initialize model to start fresh (validation run modified weights)

# Reload model
model, tokenizer = setup_model_and_tokenizer(
    model_name=config["model"]["name"],
    max_seq_length=config["model"]["max_seq_len"],
    use_unsloth=True,
)
model = apply_lora(model, config, use_unsloth=True)

# Full training arguments
train_config = config["training"]
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=train_config["num_epochs"],
    per_device_train_batch_size=train_config["per_device_train_batch_size"],
    gradient_accumulation_steps=train_config["gradient_accumulation_steps"],
    learning_rate=train_config["learning_rate"],
    lr_scheduler_type=train_config["lr_scheduler_type"],
    warmup_ratio=train_config["warmup_ratio"],
    weight_decay=train_config["weight_decay"],
    max_grad_norm=train_config["max_grad_norm"],
    logging_steps=train_config["logging_steps"],
    save_steps=train_config["save_steps"],
    save_total_limit=train_config["save_total_limit"],
    seed=SEED,
    bf16=train_config.get("bf16", True),
    optim=train_config.get("optim", "adamw_8bit"),
    report_to="wandb" if not OFFLINE else "none",
    run_name=f"lora_a_seed_{SEED}",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator,
)

print(f"Starting full training run (seed={SEED})...")
train_result = trainer.train()
print(f"Training complete. Final loss: {train_result.training_loss:.4f}")

## 5. Save Adapter

In [ ]:
# Save the final adapter
adapter_path = f"{OUTPUT_DIR}/lora_a_seed_{SEED}"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"Adapter saved to {adapter_path}")

# List saved files
!ls -la {adapter_path}

In [ ]:
# Copy to Kaggle output for download
!cp -r {adapter_path} /kaggle/working/
print("Adapter copied to /kaggle/working/ for download")